In [16]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [17]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Patch stderr/stdout fileno to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        try:
            import ipykernel.iostream
            ipykernel.iostream.OutStream.fileno = lambda self: sys.__stderr__.fileno()
        except Exception:
            sys.stderr = sys.__stderr__

In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [19]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=tools,
    system_prompt=prompt
)

In [21]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [22]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='8000f506-3070-4127-a43f-654166fd9227'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the langchain-mcp-adapters library. First, I need to check if this is a valid or existing library related to LangChain. I recall that LangChain has various integrations and adapters for different platforms, but I\'m not sure if "mcp" is one of them. Maybe MCP stands for something specific, like a platform or service.\n\nSince I don\'t have prior knowledge about this specific library, I should use the search_web tool to look for information. Let me form a query to search for "langchain-mcp-adapters library" to see if there are any official sources or documentation.\n\nAfter performing the search, I can check the results to see if there\'s an official GitHub repository, documentation, or any mentions in the La

## online

In [26]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [27]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=tools,
)

In [28]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='afa9d6cf-1d29-4ba0-b34b-656a55ac4d30'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "What time is it?" I need to figure out which function to use here. The available functions are get_current_time and convert_time. Since the user isn\'t mentioning converting time between zones, it\'s probably get_current_time. But they didn\'t specify a timezone. The default for get_current_time is America/New_York if no timezone is provided. So I should call get_current_time with the default timezone. Let me check the parameters again. The function requires a timezone, and the description says to use America/New_York if none is given. So I\'ll set timezone to America/New_York. That should give the user the current time in New York unless they specified another zone, which they didn\'t. Alright, that makes sense.\n', 'tool_calls': [{'id': '7qv